In [1]:
%matplotlib inline
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor, Lambda

training_data = datasets.FashionMNIST(
    root=r'E:\ml\dataset\cv',
    train=True,
    download=True,
    transform=ToTensor()
)

test_data = datasets.FashionMNIST(
    root=r'E:\ml\dataset\cv',
    train=False,
    download=True,
    transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
    def __init__(self):
        super(NeuralNetwork, self).__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
            nn.ReLU()
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork()

In [2]:
learning_rate = 1e-3
batch_size = 64
epochs = 5

In [3]:
loss_fn = nn.CrossEntropyLoss()

In [4]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [7]:
def train_loop(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    for batch, (X, y) in enumerate(dataloader):
        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"Loss: {loss:>7f} [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    test_loss, correct = 0, 0

    with torch.no_grad():
        for X, y in dataloader:
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= size
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [8]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

epochs = 10
for t in range(epochs):
    print(f"Epoch {t+1}\n-----------------------")
    train_loop(train_dataloader, model, loss_fn, optimizer)
    test_loop(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-----------------------
Loss: 2.285641 [    0/60000]
Loss: 2.284420 [ 6400/60000]
Loss: 2.273523 [12800/60000]
Loss: 2.268379 [19200/60000]
Loss: 2.242206 [25600/60000]
Loss: 2.265862 [32000/60000]
Loss: 2.252439 [38400/60000]
Loss: 2.248155 [44800/60000]
Loss: 2.259642 [51200/60000]
Loss: 2.244979 [57600/60000]
Test Error: 
 Accuracy: 26.5%, Avg loss: 0.035112 

Epoch 2
-----------------------
Loss: 2.254176 [    0/60000]
Loss: 2.246444 [ 6400/60000]
Loss: 2.222983 [12800/60000]
Loss: 2.221591 [19200/60000]
Loss: 2.157769 [25600/60000]
Loss: 2.210024 [32000/60000]
Loss: 2.192260 [38400/60000]
Loss: 2.178658 [44800/60000]
Loss: 2.218565 [51200/60000]
Loss: 2.181087 [57600/60000]
Test Error: 
 Accuracy: 23.7%, Avg loss: 0.034069 

Epoch 3
-----------------------
Loss: 2.210229 [    0/60000]
Loss: 2.191209 [ 6400/60000]
Loss: 2.150117 [12800/60000]
Loss: 2.153824 [19200/60000]
Loss: 2.037451 [25600/60000]
Loss: 2.130553 [32000/60000]
Loss: 2.108915 [38400/60000]
Loss: 2.085310 [4

In [9]:
torch.save(model.state_dict(), r"E:\ml\model\cv\test\fashion.pth")
print("Saved Pytorch Model State to fashion.pth")

Saved Pytorch Model State to fashion.pth
